# Section 2 — Prompt Evaluation
**Course:** Building with the Claude API — Anthropic Academy  
**Practice focus:** Eval pipeline for `PLAYWRIGHT_SYSTEM_PROMPT`

---
**Mục tiêu:** Đo chất lượng system prompt một cách khách quan thay vì chỉ test bằng mắt.

**5 bước:**
1. Setup — Load API key, tạo client
2. Eval dataset — 5 manual test inputs đa dạng
3. Run the eval — Gọi Claude với từng input
4. Code-based grading — Chấm điểm tự động bằng rule
5. Model-based grading — Dùng Claude chấm điểm output của Claude

## 1. Setup

In [ ]:
%pip install anthropic python-dotenv --quiet

In [ ]:
import os
import re
import anthropic
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
model = "claude-3-5-sonnet-latest"

# System prompt được copy từ src/prompts/playwright.prompt.ts
# Khi sửa prompt trong TypeScript, cập nhật lại đây để eval
SYSTEM_PROMPT = """
You are a senior QA automation engineer specializing in Playwright TypeScript tests.
Your job is to convert manual test steps into production-ready Playwright TypeScript code.

## Output Rules
- Always use TypeScript with async/await — never JavaScript
- Test IDs must follow format: TC-001, TC-002, TC-003...
- Every test must have at least one expect() assertion — never generate tests without assertions
- Always cover: Positive, Negative, Edge, and Security cases (minimum 4 test cases per feature)
- Add a comment on any assumption you make about the UI

## Locator Priority (follow this order)
1. getByRole() — for buttons, headings, links
2. getByLabel() — for form inputs
3. getByText() — for visible text
4. getByTestId() — only if data-testid is explicitly mentioned

## Forbidden Patterns
- No hard-coded timeouts (page.waitForTimeout) — use waitForSelector or waitForResponse instead
- No CSS selectors or XPath unless absolutely no other option
- No real passwords or PII in test data — use fake data like testuser@example.com

## Test Name Format
TC-001 [Positive] Smoke — <what action> <expected result>
TC-002 [Negative] Regression — <what action> <expected result>

## Security Test — Always Include
For any form that accepts user input, always generate at least one SQL injection or XSS test case.
"""

print("✅ Setup complete")

## 2. Eval Dataset
5 manual test inputs đa dạng — cover các loại feature khác nhau để test độ ổn định của prompt.

In [ ]:
# Mỗi item: { "id", "feature", "manual_steps" }
eval_dataset = [
    {
        "id": "DS-001",
        "feature": "Login",
        "manual_steps": """
1. Go to /login
2. Enter email testuser@example.com in the Email field
3. Enter password Test@123456 in the Password field
4. Click the Login button
5. Verify the user is redirected to /dashboard
6. Verify the page shows a welcome heading
"""
    },
    {
        "id": "DS-002",
        "feature": "Search",
        "manual_steps": """
1. Go to /products
2. Type 'laptop' in the search box
3. Click the Search button
4. Verify at least one product result is shown
5. Clear the search box
6. Click Search with empty input
7. Verify a validation message appears
"""
    },
    {
        "id": "DS-003",
        "feature": "Checkout Form",
        "manual_steps": """
1. Go to /checkout
2. Fill in Name field with 'John Doe'
3. Fill in Email field with 'john@example.com'
4. Fill in Card Number with '4242424242424242'
5. Fill in Expiry with '12/28'
6. Fill in CVV with '123'
7. Click Place Order button
8. Verify order confirmation page is shown with order number
"""
    },
    {
        "id": "DS-004",
        "feature": "File Upload",
        "manual_steps": """
1. Go to /upload
2. Click the Choose File button
3. Select a PDF file under 5MB
4. Click Upload button
5. Verify success message appears
6. Try uploading a file over 10MB
7. Verify error message about file size limit appears
"""
    },
    {
        "id": "DS-005",
        "feature": "User Profile Update",
        "manual_steps": """
1. Login and go to /profile
2. Click Edit Profile button
3. Change the Display Name to 'New Name'
4. Click Save Changes button
5. Verify success toast notification appears
6. Verify the displayed name is updated to 'New Name'
7. Try saving with empty Display Name
8. Verify validation error appears
"""
    },
]

print(f"✅ Dataset loaded: {len(eval_dataset)} test cases")
for item in eval_dataset:
    print(f"  {item['id']} — {item['feature']}")

## 2b. [Optional] Auto-Generate Dataset with Claude
Thay vì viết dataset tay, dùng **claude-haiku** (nhanh + rẻ) để tự sinh manual test steps đa dạng.  
Chạy cell này để tạo thêm test cases, sau đó thay thế `eval_dataset` ở Cell 3 bằng dataset mới.

> **Khi nào dùng?** Khi muốn test với 20-50 inputs đa dạng mà không muốn viết tay từng cái.

In [ ]:
import json

def generate_dataset(n: int = 5) -> list:
    """
    Dùng claude-haiku để tự sinh manual test steps.
    Haiku: nhanh hơn + rẻ hơn Sonnet — phù hợp cho data generation.
    """
    prompt = f"""
Generate an evaluation dataset for a QA prompt evaluation system.
The dataset will be used to test a prompt that converts manual test steps into Playwright TypeScript code.

Generate an array of {n} JSON objects. Each object represents a different web feature with realistic manual test steps.

Requirements:
- Cover diverse feature types: forms, navigation, search, file operations, user management
- Each feature should have 5-8 manual test steps written in plain English
- Steps should be realistic (how a QA manual tester would write them)
- Include both happy path and at least one error scenario per feature

Example output:
```json
[
  {{
    "id": "DS-001",
    "feature": "Login",
    "manual_steps": "1. Go to /login\\n2. Enter email...\\n3. Click Login button\\n..."
  }}
]
```

Generate {n} objects with unique features. Return ONLY the JSON array, no explanation.
"""

    messages = [{"role": "user", "content": prompt}]

    # Pre-filling trick: ép Claude bắt đầu bằng ```json
    messages.append({"role": "assistant", "content": "```json"})

    response = client.messages.create(
        model="claude-haiku-4-5",   # Haiku: nhanh + rẻ cho data generation
        max_tokens=3000,
        temperature=0.7,            # Temperature cao hơn = dataset đa dạng hơn
        stop_sequences=["```"],     # Dừng trước closing fence → lấy JSON sạch
        messages=messages,
    )

    raw = response.content[0].text.strip()
    return json.loads(raw)


# Sinh dataset tự động — chạy 1 lần, lưu vào file để tái sử dụng
print("Generating dataset with Claude Haiku...")
auto_dataset = generate_dataset(n=5)

# Lưu vào file để không phải gọi API lại mỗi lần
dataset_path = "../notebooks/eval_dataset.json"
with open(dataset_path, "w", encoding="utf-8") as f:
    json.dump(auto_dataset, f, indent=2, ensure_ascii=False)

print(f"✅ Generated {len(auto_dataset)} test cases → saved to {dataset_path}\n")
for item in auto_dataset:
    print(f"  {item['id']} — {item['feature']}")

# Uncomment dòng dưới để dùng auto dataset thay cho dataset viết tay ở Cell 3:
# eval_dataset = auto_dataset


## 3. Run the Eval
Gọi Claude với từng manual test input và thu thập kết quả.

In [ ]:
def generate_playwright_test(manual_steps: str) -> str:
    """Gọi Claude để convert manual steps thành Playwright code."""
    response = client.messages.create(
        model=model,
        max_tokens=4000,
        temperature=0.1,   # Low temperature = nhất quán, phù hợp cho code gen
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": f"Convert the following manual test steps into a Playwright TypeScript test:\n\n{manual_steps}"}]
    )
    return response.content[0].text


# Chạy eval — gọi Claude cho từng item trong dataset
print("Running eval...\n")
results = []

for item in eval_dataset:
    print(f"⏳ {item['id']} — {item['feature']}...", end=" ")
    output = generate_playwright_test(item["manual_steps"])
    results.append({
        "id": item["id"],
        "feature": item["feature"],
        "manual_steps": item["manual_steps"],
        "output": output,
    })
    print("✅")

print(f"\n✅ Eval complete — {len(results)} outputs collected")

## 4. Code-Based Grading
Chấm điểm tự động bằng 2 lớp — nhanh, không tốn API call.

**Lớp 1 — Rule checks (6 điểm):** Format, coverage, best practices  
**Lớp 2 — Syntax validation (4 điểm):** Parse TypeScript để phát hiện lỗi cú pháp

| Rule | Điểm | Lý do |
|---|---|---|
| Có `TC-001` format | +1 | QA standard |
| Có `expect(` | +1 | Không có assertion = test vô nghĩa |
| Có `[Positive]` và `[Negative]` | +1 | Coverage tối thiểu |
| Có security test | +1 | Bắt buộc theo CLAUDE.md |
| Không có `waitForTimeout` | +1 | Best practice Playwright |
| Có `import { test, expect }` | +1 | TypeScript import chuẩn |
| **Syntax valid** (no obvious errors) | **+4** | Code phải parse được |

In [ ]:
import subprocess
import tempfile
import os

def validate_typescript_syntax(code: str) -> tuple[bool, str]:
    """
    Validate TypeScript syntax bằng cách chạy tsc --noEmit.
    Tương tự validate_python() dùng ast.parse() trong bài học,
    nhưng dành cho TypeScript.
    Returns: (is_valid, error_message)
    """
    # Trích xuất code block từ markdown nếu có
    ts_code = code
    if "```typescript" in code:
        start = code.find("```typescript") + len("```typescript")
        end = code.find("```", start)
        ts_code = code[start:end].strip() if end != -1 else code[start:].strip()
    elif "```ts" in code:
        start = code.find("```ts") + len("```ts")
        end = code.find("```", start)
        ts_code = code[start:end].strip() if end != -1 else code[start:].strip()

    # Heuristic checks thay vì chạy tsc (không cần cài TypeScript)
    errors = []

    # Kiểm tra unmatched braces
    if ts_code.count("{") != ts_code.count("}"):
        errors.append(f"Unmatched braces: {ts_code.count('{')} open, {ts_code.count('}')} close")

    # Kiểm tra unmatched parentheses
    if ts_code.count("(") != ts_code.count(")"):
        errors.append(f"Unmatched parentheses")

    # Kiểm tra có async/await pattern đúng không
    if "async ({" in ts_code and "await " not in ts_code:
        errors.append("async function without await")

    # Kiểm tra test.describe wrapper
    if "test(" in ts_code and "test.describe(" not in ts_code:
        errors.append("Missing test.describe() wrapper")

    is_valid = len(errors) == 0
    return is_valid, "; ".join(errors) if errors else "OK"


def validate_syntax_score(output: str) -> tuple[int, str]:
    """
    Syntax validation — trả về score 0 hoặc 4.
    Giống validate_json()/validate_python() trong bài học:
    - Valid = 4 điểm (tỉ lệ cao để penalize lỗi syntax nặng)
    - Invalid = 0 điểm
    """
    is_valid, error = validate_typescript_syntax(output)
    return (4, "OK") if is_valid else (0, error)


def code_grade(output: str) -> dict:
    """
    Chấm điểm output bằng 2 lớp:
    - Rule checks: 6 điểm
    - Syntax validation: 4 điểm
    Tổng tối đa: 10 điểm
    """
    score = 0
    checks = {}

    # ── Lớp 1: Rule checks (6 điểm) ────────────────────────────────────────
    checks["tc_format"]       = bool(re.search(r"TC-\d{3}", output))
    checks["has_assertion"]   = "expect(" in output
    checks["has_coverage"]    = "[Positive]" in output and "[Negative]" in output
    checks["has_security"]    = bool(re.search(r"(injection|XSS|security|SQL)", output, re.IGNORECASE))
    checks["no_hard_timeout"] = "waitForTimeout" not in output
    checks["has_import"]      = "import { test, expect }" in output or "from '@playwright/test'" in output

    for key in ["tc_format", "has_assertion", "has_coverage",
                "has_security", "no_hard_timeout", "has_import"]:
        if checks[key]:
            score += 1

    # ── Lớp 2: Syntax validation (4 điểm) ───────────────────────────────────
    syntax_score, syntax_error = validate_syntax_score(output)
    checks["syntax_valid"] = syntax_score == 4
    checks["syntax_error"] = syntax_error
    score += syntax_score

    return {"score": score, "checks": checks}


# Chấm điểm tất cả outputs
print(f"{'ID':<8} {'Feature':<22} {'Score':>5}  {'TC':>3} {'Assert':>6} {'Cover':>5} {'Sec':>4} {'Syntax':>7}")
print("-" * 70)

scores = []
for r in results:
    grade = code_grade(r["output"])
    scores.append(grade["score"])
    c = grade["checks"]
    syntax_icon = "✅" if c["syntax_valid"] else "❌"
    print(f"{r['id']:<8} {r['feature']:<22} {grade['score']:>4}/10  "
          f"{'✅' if c['tc_format'] else '❌':>3} "
          f"{'✅' if c['has_assertion'] else '❌':>6} "
          f"{'✅' if c['has_coverage'] else '❌':>5} "
          f"{'✅' if c['has_security'] else '❌':>4} "
          f"{syntax_icon:>7}")
    if not c["syntax_valid"]:
        print(f"         ⚠️  Syntax error: {c['syntax_error']}")

avg = sum(scores) / len(scores)
print("-" * 70)
print(f"{'Average code score':<30} {avg:.1f}/10")


## 5. Model-Based Grading
Dùng Claude chấm điểm output của Claude — bắt được lỗi logic mà code-based không thấy được.

> **Lưu ý:** Tốn thêm API call (1 call grading cho mỗi output). Chỉ chạy khi cần đánh giá sâu.

In [ ]:
from statistics import mean

GRADER_PROMPT = """
You are a QA lead reviewing AI-generated Playwright TypeScript tests.
Score the test output from 1 to 10 based on these criteria:

- **Completeness** (1-3): Does it cover Positive, Negative, Edge, and Security cases?
- **Locator quality** (1-3): Does it use getByRole/getByLabel/getByText? Penalize CSS selectors or XPath.
- **Assertion quality** (1-2): Are expect() assertions meaningful and correct?
- **Code quality** (1-2): Is the code clean, readable, and following Playwright best practices?

IMPORTANT: Do NOT default to middling scores like 5 or 6.
- Give 9-10 only when the output is genuinely excellent with no clear improvements
- Give 1-3 when the output has critical failures (no assertions, wrong format, CSS selectors)
- Be specific in your reasoning — vague scores are not useful

Respond with ONLY a JSON object in this exact format (no other text):
{
  "score": <1-10>,
  "strengths": "<what the output does well>",
  "weaknesses": "<what is missing or wrong>",
  "reasoning": "<why this specific score>"
}
"""

def model_grade(manual_steps: str, output: str) -> dict:
    """
    Dùng Claude chấm điểm output — hỏi cả strengths/weaknesses/reasoning.
    Pre-filling + stop_sequences để lấy JSON sạch (kỹ thuật từ bài Structured Data).
    """
    messages = [
        {
            "role": "user",
            "content": f"Manual steps:\n{manual_steps}\n\nGenerated Playwright test:\n{output}"
        },
        # Pre-filling: ép Claude bắt đầu bằng JSON object
        {
            "role": "assistant",
            "content": "{"
        }
    ]

    response = client.messages.create(
        model=model,
        max_tokens=400,
        temperature=0.0,        # Temperature 0 = grader nhất quán, không random
        system=GRADER_PROMPT,
        stop_sequences=["}"],   # Dừng sau closing brace
        messages=messages,
    )

    import json
    # Ghép lại phần bị cắt bởi pre-filling và stop_sequence
    raw = "{" + response.content[0].text.strip() + "}"
    return json.loads(raw)


def run_test_case(r: dict, code_score: int) -> dict:
    """Chạy model grading cho một test case và trả về kết quả đầy đủ."""
    grade = model_grade(r["manual_steps"], r["output"])
    return {
        "id": r["id"],
        "feature": r["feature"],
        "code_score": code_score,
        "model_score": grade["score"],
        "strengths": grade.get("strengths", ""),
        "weaknesses": grade.get("weaknesses", ""),
        "reasoning": grade.get("reasoning", ""),
    }


# Chạy model-based grading cho toàn bộ dataset
print("Running model-based grading...\n")
graded_results = []
for i, r in enumerate(results):
    print(f"⏳ Grading {r['id']} — {r['feature']}...", end=" ")
    result = run_test_case(r, scores[i])
    graded_results.append(result)
    print(f"✅ {result['model_score']}/10")

# Tổng hợp kết quả
print(f"\n{'─'*80}")
print(f"{'ID':<8} {'Feature':<22} {'Code':>5} {'Model':>6}  Weakness")
print(f"{'─'*80}")
for r in graded_results:
    weakness_short = r["weaknesses"][:40] if r["weaknesses"] else "—"
    print(f"{r['id']:<8} {r['feature']:<22} {r['code_score']:>4}/10 {r['model_score']:>5}/10  {weakness_short}")

avg_model = mean([r["model_score"] for r in graded_results])
avg_code  = mean([r["code_score"]  for r in graded_results])
print(f"{'─'*80}")
print(f"{'Average — Code grading':<30} {avg_code:.1f}/10")
print(f"{'Average — Model grading':<30} {avg_model:.1f}/10")

# In chi tiết reasoning cho output điểm thấp nhất (để biết cần cải thiện gì)
lowest = min(graded_results, key=lambda x: x["model_score"])
print(f"\n⚠️  Lowest score: {lowest['id']} — {lowest['feature']} ({lowest['model_score']}/10)")
print(f"   Weaknesses : {lowest['weaknesses']}")
print(f"   Reasoning  : {lowest['reasoning']}")


## 6. Iterate — Sửa Prompt và So Sánh
Sau khi có baseline score, sửa `SYSTEM_PROMPT` ở Cell 2 rồi chạy lại từ Cell 3 để xem score thay đổi.

```
Prompt v1 (baseline) → score X.X/10
Prompt v2 (thêm rule) → score Y.Y/10  ← tốt hơn hay tệ hơn?
```

> **Tip:** Dùng `Kernel > Restart and Run All` để chạy lại toàn bộ eval với prompt mới.

In [ ]:
from statistics import mean

# ── Combined Score: Code grade + Model grade ────────────────────────────────
# Theo bài học: final_score = (code_score + model_score) / 2
# Code grade: syntax + format rules (objective)
# Model grade: quality + completeness + reasoning (subjective)

print("=" * 70)
print("FINAL COMBINED SCORES")
print("=" * 70)
print(f"{'ID':<8} {'Feature':<22} {'Code':>5} {'Model':>6} {'Final':>6}")
print("-" * 70)

final_scores = []
for code_r, model_r in zip(results, graded_results):
    code_s  = code_grade(code_r["output"])["score"]
    model_s = model_r["model_score"]
    final   = (code_s + model_s) / 2
    final_scores.append(final)
    print(f"{code_r['id']:<8} {code_r['feature']:<22} {code_s:>4}/10 {model_s:>5}/10 {final:>5.1f}/10")

print("-" * 70)
print(f"{'AVERAGE FINAL SCORE':<38} {mean(final_scores):>5.1f}/10")
print("=" * 70)
print("\n📊 Interpretation:")
print(f"  9-10 : Production-ready — excellent quality")
print(f"  7-8  : Good — minor improvements needed")
print(f"  5-6  : Acceptable — prompt needs refinement")
print(f"  < 5  : Poor — significant issues, revise SYSTEM_PROMPT")
